In [1]:
# Auditoria de Segurança - Scytalone vs Humano
import subprocess
import pandas as pd
from io import StringIO

# --- CONFIGURAÇÃO ---
# Coloque aqui o ID exato da Scytalone que está na lista Platinum
# Exemplo (Verifique se o ID é este mesmo na sua lista):
ID_SCYTALONE = "lcl|SDAQ01000016.1_cds_KAI3555617.1_3992" 

ARQUIVO_FASTA = "c_abscissum_data/ncbi_dataset/data/GCA_023376855.1/cds_from_genomic.fna"
DB_HUMANO = "db_human"

# --- 1. RECUPERAR A SEQUÊNCIA ---
from Bio import SeqIO
seq_scytalone = ""
for record in SeqIO.parse(ARQUIVO_FASTA, "fasta"):
    if record.id == ID_SCYTALONE:
        seq_scytalone = str(record.seq)
        break

if not seq_scytalone:
    print("ERRO: ID da Scytalone não encontrado no FASTA.")
else:
    print(f"Auditando gene: {ID_SCYTALONE}")
    print(f"Tamanho: {len(seq_scytalone)} pb")
    
    # --- 2. RODAR BLAST SEM FILTROS RÍGIDOS ---
    # Vamos pedir para mostrar TUDO, até matches pequenos de 15pb
    with open("temp_audit.fasta", "w") as f:
        f.write(f">{ID_SCYTALONE}\n{seq_scytalone}")
        
    cmd = [
        "blastn",
        "-task", "blastn-short",
        "-query", "temp_audit.fasta",
        "-db", DB_HUMANO,
        "-outfmt", "6 qseqid sseqid pident length qstart qend sstart send evalue mismatches gapopen",
        "-word_size", "7",   # Semente pequena para achar tudo
        "-evalue", "1000"
    ]
    
    print("\nRodando BLAST contra Humano (Parâmetros relaxados)...")
    result = subprocess.run(cmd, capture_output=True, text=True)
    
    if result.stdout:
        colunas = ["qseqid", "sseqid", "pident", "length", "qstart", "qend", "sstart", "send", "evalue", "mismatches", "gapopen"]
        df = pd.read_csv(StringIO(result.stdout), sep="\t", names=colunas)
        
        # Ordenar pelos matches mais longos
        df = df.sort_values(by="length", ascending=False)
        
        print(f"\nForam encontrados {len(df)} alinhamentos locais.")
        print("Mostrando os Top 5 matches mais perigosos:")
        print(df[['sseqid', 'length', 'pident', 'mismatches']].head(5))
        
        # --- VEREDITO ---
        maior_match = df.iloc[0]
        print("\n--- VEREDITO DA AUDITORIA ---")
        print(f"Maior match encontrado: {maior_match['length']} pb")
        print(f"Identidade: {maior_match['pident']}%")
        
        if maior_match['length'] >= 21 and maior_match['pident'] == 100:
            print("🔴 CRÍTICO: Existe um match de >=21pb idêntico. O SCRIPT DE BATCH FALHOU.")
        elif maior_match['length'] >= 21 and maior_match['pident'] < 100:
            print("🟢 SEGURO: O match é longo, mas tem mutações (não é 100%). O RNAi falha.")
        elif maior_match['length'] < 21:
            print("🟢 SEGURO: O match é 100% igual, mas é muito curto (menor que 21pb). O RISC não cliva.")
            
    else:
        print("Nenhum match encontrado, nem mesmo parcial.")

Auditando gene: lcl|SDAQ01000016.1_cds_KAI3555617.1_3992
Tamanho: 576 pb

Rodando BLAST contra Humano (Parâmetros relaxados)...

Foram encontrados 501 alinhamentos locais.
Mostrando os Top 5 matches mais perigosos:
                                        sseqid  length  pident  mismatches
364  lcl|NC_000007.14_cds_XP_016867341.1_52035      23  91.304           0
363  lcl|NC_000007.14_cds_NP_001274079.1_52036      23  91.304           0
362  lcl|NC_000007.14_cds_NP_001104508.1_52037      23  91.304           0
355  lcl|NC_000007.14_cds_XP_047276000.1_52044      23  91.304           0
356  lcl|NC_000007.14_cds_XP_006715961.1_52043      23  91.304           0

--- VEREDITO DA AUDITORIA ---
Maior match encontrado: 23 pb
Identidade: 91.304%
🟢 SEGURO: O match é longo, mas tem mutações (não é 100%). O RNAi falha.
